In [ ]:
!pip install torch torchvision pillow

# Import Library

In [ ]:
from pathlib import Path
import csv
import os
import urllib.request

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

import torch
from torchvision import models, transforms
from PIL import Image

# Path folder

In [ ]:
CSV_PATH = Path("dataset/labels.csv")
IMAGE_DIR = Path("dataset/images")
MODEL_DIR = Path("models")

MODEL_DIR.mkdir(exist_ok=True)

print("CSV input:", CSV_PATH)
print("Folder gambar:", IMAGE_DIR)
print("Folder model:", MODEL_DIR)

# Download label Places365

In [ ]:
LABEL_URL = "https://raw.githubusercontent.com/CSAILVision/places365/master/categories_places365.txt"
LABEL_PATH = MODEL_DIR / "categories_places365.txt"

if not LABEL_PATH.exists():
    print("Downloading labels...")
    urllib.request.urlretrieve(LABEL_URL, LABEL_PATH)
    print("Label selesai didownload.")
else:
    print("Label sudah ada.")

classes = []

with open(LABEL_PATH, "r") as f:
    for line in f:
        classes.append(line.strip().split(" ")[0][3:])

print("Jumlah scene:", len(classes))
print("Contoh label:", classes[:10])

# Download model ResNet50-Places365

In [ ]:
MODEL_URL = "http://places2.csail.mit.edu/models_places365/resnet50_places365.pth.tar"
MODEL_PATH = MODEL_DIR / "resnet50_places365.pth.tar"

if not MODEL_PATH.exists():
    print("Downloading ResNet50-Places365...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("Model selesai didownload.")
else:
    print("Model sudah ada.")

# Load model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device yang dipakai:", device)

model = models.resnet50(num_classes=365)

checkpoint = torch.load(MODEL_PATH, map_location=device)
state_dict = checkpoint["state_dict"]

new_state_dict = {}

for key, value in state_dict.items():
    new_key = key.replace("module.", "")
    new_state_dict[new_key] = value

model.load_state_dict(new_state_dict)
model.to(device)
model.eval()

print("Model ResNet50-Places365 berhasil dimuat.")

# Preprocessing gambar

In [ ]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Fungsi prediksi Top-K scene

In [ ]:
def predict_scene(image_path, topk=5):
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(f"Gambar tidak ditemukan: {image_path}")

    image = Image.open(image_path).convert("RGB")
    topk = min(topk, len(classes))
    
    input_tensor = transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(input_tensor)
        probabilities = torch.nn.functional.softmax(output[0], dim=0)
    
    top_probs, top_indices = torch.topk(probabilities, topk)
    
    results = []
    
    for prob, idx in zip(top_probs, top_indices):
        scene_name = classes[idx.item()].replace("_", " ")
        results.append((scene_name, prob.item()))
    
    return image, results

# Update CSV dengan prediksi scene

In [ ]:
def find_image_path(file_name, image_dir=IMAGE_DIR):
    clean_name = Path(str(file_name).strip()).name
    if not clean_name:
        return None

    image_path = image_dir / clean_name
    if image_path.exists():
        return image_path

    candidates = sorted(image_dir.glob(f"{Path(clean_name).stem}.*"))
    return candidates[0] if candidates else None


def predict_top_scene(image_path):
    _, results = predict_scene(image_path, topk=1)
    return results[0][0] if results else ""


def update_csv_with_scene_predictions(csv_path=CSV_PATH, image_dir=IMAGE_DIR, output_column="prediction_sence"):
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV tidak ditemukan: {csv_path}")

    with csv_path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        fieldnames = list(reader.fieldnames or [])

    if "File" not in fieldnames:
        raise KeyError("Kolom 'File' tidak ditemukan di CSV.")

    if output_column not in fieldnames:
        fieldnames.append(output_column)

    total_rows = len(rows)
    predicted_count = 0
    missing_count = 0
    error_count = 0

    for index, row in enumerate(rows, start=1):
        image_path = find_image_path(row.get("File", ""), image_dir=image_dir)

        if image_path is None:
            row[output_column] = ""
            missing_count += 1
        else:
            try:
                row[output_column] = predict_top_scene(image_path)
                predicted_count += 1
            except Exception as exc:
                row[output_column] = ""
                error_count += 1
                print(f"Error saat prediksi {image_path.name}: {exc}")

        if index == 1 or index % 50 == 0 or index == total_rows:
            print(f"Processed {index}/{total_rows} rows")

    temp_path = csv_path.with_suffix(csv_path.suffix + ".tmp")
    with temp_path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    temp_path.replace(csv_path)

    print("\nCSV berhasil diupdate:", csv_path)
    print("Berhasil diprediksi:", predicted_count)
    print("Gambar tidak ditemukan:", missing_count)
    print("Gagal diprediksi:", error_count)

    return rows

# Proses Gambar

In [ ]:
updated_rows = update_csv_with_scene_predictions()

print("\nPreview hasil:")
for row in updated_rows[:10]:
    print(row["File"], "->", row["prediction_sence"])